# 01. EDA — 엔진 센서 데이터 탐색

**목표:** 센서 21개 중 엔진 열화(고장 진행)를 잘 보여주는 센서 찾기

- 실행: 셀을 클릭하고 `Shift + Enter`
- 데이터 위치: `../data/train_FD001.txt`

## Step 1. 데이터 불러오기

In [ ]:
import pandas as pd

# 원본 파일에는 컬럼 이름이 없어서 직접 붙여준다
cols = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

df = pd.read_csv("../data/train_FD001.txt", sep=r"\s+", header=None, names=cols)
df.head()

## Step 2. 기본 정보 확인

In [ ]:
print("크기 (행, 열):", df.shape)          # (20631, 26)이 나오면 성공
print("엔진 수:", df["unit"].nunique())
print("빈칸 개수:", df.isnull().sum().sum())

In [ ]:
# 엔진별 수명(마지막 사이클) 통계
df.groupby("unit")["cycle"].max().describe()

## Step 3. RUL(남은 수명) 계산

`RUL = 그 엔진의 마지막 사이클 - 현재 사이클`

In [ ]:
max_cycle = df.groupby("unit")["cycle"].transform("max")
df["RUL"] = max_cycle - df["cycle"]
df[["unit", "cycle", "RUL"]].head()

## Step 4. 값이 거의 안 변하는 센서 찾기

표준편차(std)가 0에 가까우면 → 값이 거의 안 변함 → 정보가 없음

In [ ]:
sensor_cols = [f"s{i}" for i in range(1, 22)]
df[sensor_cols].std().sort_values()

## Step 5. 그래프로 확인

- 왼쪽 = 새 엔진, 오른쪽 = 고장 직전 (RUL 0)
- 오른쪽으로 갈수록 올라가거나 내려가면 → 좋은 센서

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(7, 3, figsize=(15, 20))
for ax, s in zip(axes.flat, sensor_cols):
    for unit in [1, 2, 3, 4, 5]:          # 엔진 5대만 그리기
        d = df[df["unit"] == unit]
        ax.plot(d["RUL"], d[s], alpha=0.7)
    ax.set_title(s)
    ax.invert_xaxis()
plt.tight_layout()
plt.show()

## Step 6. RUL과의 상관관계

In [ ]:
corr = df[sensor_cols + ["RUL"]].corr()["RUL"].drop("RUL")
corr.abs().sort_values(ascending=False)

## ✅ 정리 (직접 채우기)

| 구분 | 센서 |
|---|---|
| ✅ 사용할 센서 | |
| ❌ 버릴 센서 | |

**발견한 점 메모:**
- 